In [ ]:
import scanpy as sc
import anndata as ad
from pathlib import Path
from geneformer import TranscriptomeTokenizer

In [17]:
input_h5ad = '/projects/shared/intronic_bam/datasets/anndata/be1_scGPT_embeddings.h5ad'
prep_dir = "/projects/shared/intronic_bam/datasets/geneformer/prep_be1"
output_dir = "/projects/shared/intronic_bam/datasets/geneformer"
output_prefix = "be1"

In [18]:
Path(prep_dir).mkdir(parents=True, exist_ok=True)
Path(output_dir).mkdir(parents=True, exist_ok=True)
prepped_h5ad = Path(prep_dir) / "be1_prepped.h5ad"

In [19]:
print(f"Loading original dataset: {input_h5ad}")
adata = sc.read_h5ad(input_h5ad)

Loading original dataset: /projects/shared/intronic_bam/datasets/anndata/be1_scGPT_embeddings.h5ad


In [20]:
adata

AnnData object with n_obs × n_vars = 29128 × 24288
    obs: 'Sample', 'Barcode', 'sum', 'detected', 'subsets_Mito_sum', 'subsets_Mito_detected', 'subsets_Mito_percent', 'total', 'discard', 'is_train', 'dataset', 'is_ref', 'Sample_masked', 'predicted_cell_type', 'n_counts'
    var: 'ID', 'Symbol', 'Type', 'gene_names', 'id_in_vocab', 'ensembl_id'
    uns: 'Sample_masked_colors', 'is_ref_colors', 'neighbors', 'umap'
    obsm: 'X_scGPT', 'X_umap'
    layers: 'counts'
    obsp: 'connectivities', 'distances'

In [21]:
print("Formatting metadata for Geneformer...")
# Geneformer TranscriptomeTokenizer hardcodes the requirement for:
# 1. 'ensembl_id' in adata.var
# 2. 'n_counts' in adata.obs
    
# Assuming 'ID' contains the Ensembl IDs. If it's 'Symbol' or another column, adjust here.
adata.var['ensembl_id'] = adata.var['ID']

# Calculate the total read counts per cell
adata.obs['n_counts'] = adata.X.sum(axis=1)

Formatting metadata for Geneformer...


In [22]:
print(f"Saving prepped dataset to {prepped_h5ad}")
# Save the prepared anndata
adata.write_h5ad(prepped_h5ad)

Saving prepped dataset to /projects/shared/intronic_bam/datasets/geneformer/prep_be1/be1_prepped.h5ad


In [23]:
# Now we can tokenize
print("Initializing TranscriptomeTokenizer...")
# Add custom attributes we want to retain in the tokenized dataset.
# We map output_name: input_name
# From earlier notebooks, 'Sample' is the cell type key for BE1.
tk = TranscriptomeTokenizer(
    custom_attr_name_dict={"Sample": "Sample", "Barcode": "Barcode"}, 
    nproc=10
)

Initializing TranscriptomeTokenizer...


In [24]:
!ls /projects/shared/intronic_bam/datasets/geneformer/prep_be1/

be1_prepped.h5ad


In [25]:
print("Tokenizing data...")
tk.tokenize_data(
    data_directory=prep_dir,
    output_directory=output_dir,
    output_prefix=output_prefix,
    file_format="h5ad",
    use_generator=False
)
    
print(f"Tokenization complete! Tokenized dataset saved at: {Path(output_dir) / (output_prefix + '.dataset')}")


Tokenizing data...
Tokenizing /projects/shared/intronic_bam/datasets/geneformer/prep_be1/be1_prepped.h5ad


100%|██████████| 57/57 [00:18<00:00,  3.14it/s]
/home/vreffo/micromamba/envs/nw-huggingface/python/lib/python3.11/site-packages/geneformer/tokenizer.py:544: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  for i in adata.var["ensembl_id_collapsed"][coding_miRNA_loc]
/home/vreffo/micromamba/envs/nw-huggingface/python/lib/python3.11/site-packages/geneformer/tokenizer.py:547: FutureWarning: Series.__getitem__ treating keys as positions is deprecated. In a future version, integer keys will always be treated as labels (consistent with DataFrame behavior). To access a value by position, use `ser.iloc[pos]`
  coding_miRNA_ids = adata.var["ensembl_id_collapsed"][coding_miRNA_loc]


/projects/shared/intronic_bam/datasets/geneformer/prep_be1/be1_prepped.h5ad has no column attribute 'filter_pass'; tokenizing all cells.
Creating dataset.
Tokenization complete! Tokenized dataset saved at: /projects/shared/intronic_bam/datasets/geneformer/be1.dataset


In [2]:
from geneformer import EmbExtractor

/home/vreffo/micromamba/envs/nw-huggingface/python/lib/python3.11/site-packages/transformers/utils/hub.py:128: FutureWarning: Using `TRANSFORMERS_CACHE` is deprecated and will be removed in v5 of Transformers. Use `HF_HOME` instead.
  warnings.warn(


In [3]:
# Define paths for the model, tokenized input data, and output directory
# Note: Geneformer expects the input data to be pre-tokenized into a .dataset folder.
# Replace these paths with the actual paths to your Geneformer model and tokenized BE1 dataset.
model_dir = "/software/huggingface/hub/models--ctheodoris--Geneformer/snapshots/04c2b2e84da7c0f385c3f9ad8f3ec24bab6650e5/Geneformer-V2-104M/"
input_data_dir = "/projects/shared/intronic_bam/datasets/geneformer/be1.dataset"
output_dir = "/projects/shared/intronic_bam/datasets/embeddings/geneformer"
output_prefix = "be1_geneformer"

In [4]:
# Ensure output directory exists
Path(output_dir).mkdir(parents=True, exist_ok=True)

NameError: name 'Path' is not defined

In [5]:
# Initialize the EmbExtractor
# The BE1 dataset uses "Sample" for its cell type labels.
embex = EmbExtractor(
    model_type="Pretrained",     # Use "CellClassifier" if you fine-tuned a model
    num_classes=7,               # Set to number of classes if using a classifier
    emb_mode="cell",
    cell_emb_style="mean_pool",
    filter_data=None,            # Add filtering dict if needed, e.g., {"Sample": ["some_sample"]}
    max_ncells=None,             # Set to a number (e.g., 1000) to downsample, or None for all cells
    emb_layer=-1,                # -1 extracts the 2nd to last layer (recommended for pretrained)
    emb_label=["Barcode", "Sample"], # Label to append to output and use for plotting
    labels_to_plot=["Sample"],   # Label used to color UMAP and Heatmap
    forward_batch_size=8,       # Adjust based on your GPU memory
    nproc=10,                    # Number of CPU processes
    summary_stat=None            # Output full embeddings. Change to "mean"/"median" if memory constrained
)

In [ ]:
# Extract the embeddings
print("Extracting Geneformer embeddings for BE1 dataset...")
embs_df = embex.extract_embs(
    model_directory=model_dir,
    input_data_file=input_data_dir,
    output_directory=output_dir,
    output_prefix=output_prefix,
    output_torch_embs=False
)
    
print(f"Embeddings extracted and saved to {output_dir}/{output_prefix}.csv")
print(f"Embeddings DataFrame shape: {embs_df.shape}")


BertForMaskedLM has generative capabilities, as `prepare_inputs_for_generation` is explicitly overwritten. However, it doesn't directly inherit from `GenerationMixin`. From 👉v4.50👈 onwards, `PreTrainedModel` will NOT inherit from `GenerationMixin`, and this model will lose the ability to call `generate` and other related functions.
  - If you're using `trust_remote_code=True`, you can get rid of this warning by loading the model with an auto class. See https://huggingface.co/docs/transformers/en/model_doc/auto#auto-classes
  - If you are the owner of the model architecture code, please modify your model class such that it inherits from `GenerationMixin` (after `PreTrainedModel`, otherwise you'll get an exception).
  - If you are not the owner of the model architecture class, please contact the model code owner to update it.


Extracting Geneformer embeddings for BE1 dataset...


/home/vreffo/micromamba/envs/nw-huggingface/python/lib/python3.11/site-packages/transformers/generation/configuration_utils.py:774: UserWarning: `return_dict_in_generate` is NOT set to `True`, but `output_hidden_states` is. When `return_dict_in_generate` is not `True`, `output_hidden_states` is ignored.
  warnings.warn(
CLS token present in token dictionary, excluding from average.
EOS token present in token dictionary, excluding from average.


  0%|          | 0/3641 [00:00<?, ?it/s]

In [ ]:
adata

In [ ]:
# Plot UMAP and Heatmap of the embeddings colored by Sample
print("Plotting embeddings...")
embex.plot_embs(
    embs=embs_df, 
    plot_style="umap",
    output_directory=output_dir,
    output_prefix=output_prefix,
    max_ncells_to_plot=5000,     # Downsample for plotting if the dataset is large
    kwargs_dict={"palette": "Set1", "size": 50}
)